# 📥 Import Price History - Jupyter Notebook

**Import:** 208,700 rows  
**เวลา:** 1-2 นาที

---

## ขั้นตอน:

1. **Cell 1:** ตั้งค่า MySQL
2. **Cell 2:** เชื่อมต่อและเช็คข้อมูล
3. **Cell 3:** Import price_history
4. **Cell 4:** ตรวจสอบผลลัพธ์

**⚠️ สำคัญ:** ต้อง import etf_master, benchmark_portfolios, benchmark_holdings ก่อน!

---

## Cell 1: ตั้งค่า MySQL

**แก้ password ให้ตรงกับเครื่องคุณ**

In [ ]:
# ========================================
# ตั้งค่า MySQL Connection
# ========================================

MYSQL_CONFIG = {
    'host': '127.0.0.1',
    'port': 3306,
    'user': 'root',
    'password': 'krittanut123456',  # 🔐 แก้ไข password ตรงนี้
    'database': 'portfolio_backtesting'
}

# ========================================
# ติดตั้ง libraries (ถ้ายังไม่มี)
# ========================================

try:
    import mysql.connector
    import pandas as pd
    from datetime import datetime
    print("✅ Libraries พร้อมใช้งาน")
except ImportError:
    print("⏳ กำลังติดตั้ง libraries...")
    import sys
    !{sys.executable} -m pip install mysql-connector-python pandas -q
    import mysql.connector
    import pandas as pd
    from datetime import datetime
    print("✅ ติดตั้ง libraries เสร็จแล้ว")

print(f"\n✅ ตั้งค่าเรียบร้อย")
print(f"   MySQL: {MYSQL_CONFIG['user']}@{MYSQL_CONFIG['host']}:{MYSQL_CONFIG['port']}/{MYSQL_CONFIG['database']}")

---

## Cell 2: เชื่อมต่อ MySQL และเช็คข้อมูล

In [ ]:
print("=" * 70)
print("🔌 เชื่อมต่อ MySQL...")
print("=" * 70)

# เชื่อมต่อ MySQL
try:
    conn = mysql.connector.connect(**MYSQL_CONFIG)
    cursor = conn.cursor(dictionary=True)
    print("✅ เชื่อมต่อสำเร็จ\n")
except mysql.connector.Error as e:
    print(f"❌ Error: {e}\n")
    print("💡 กรุณาตรวจสอบ:")
    print("   1. MySQL server ทำงานอยู่หรือไม่?")
    print("   2. Password ถูกต้องหรือไม่? (แก้ใน Cell 1)")
    print("   3. Database 'portfolio_backtesting' มีอยู่หรือไม่?")
    raise

# ========================================
# ตรวจสอบข้อมูลที่มีอยู่
# ========================================
print("📋 ตรวจสอบข้อมูลที่มีอยู่:")
print("=" * 70)

tables_to_check = ['etf_master', 'benchmark_portfolios', 'benchmark_holdings', 'price_history']
expected = [50, 35, 114, 0]

all_ok = True
for table, expect in zip(tables_to_check, expected):
    cursor.execute(f"SELECT COUNT(*) as cnt FROM {table}")
    count = cursor.fetchone()['cnt']
    
    if table == 'price_history':
        status = "⏳" if count == 0 else "✅"
    else:
        status = "✅" if count >= expect * 0.9 else "❌"
        if count < expect * 0.9:
            all_ok = False
    
    print(f"{status} {table:25s}: {count:>10,} rows (expect ~{expect:,})")

print("=" * 70)

if not all_ok:
    print("\n❌ ยังไม่มีข้อมูล etf_master, benchmark_portfolios, หรือ benchmark_holdings")
    print("\n💡 กรุณารัน IMPORT_ALL_DATA.sql ก่อน:")
    print("   1. เปิดไฟล์ IMPORT_ALL_DATA.sql")
    print("   2. Copy ทั้งหมด (Ctrl+A, Ctrl+C)")
    print("   3. Paste ใน MySQL Workbench")
    print("   4. Execute (Ctrl+Shift+Enter)")
    raise Exception("ต้อง import 3 ตารางแรกก่อน")

# Get ETF mapping
cursor.execute("SELECT etf_id, ticker_symbol FROM etf_master ORDER BY ticker_symbol")
etf_map = {r['ticker_symbol']: r['etf_id'] for r in cursor.fetchall()}
print(f"\n✅ สร้าง mapping: {len(etf_map)} tickers → IDs")

# ตัวอย่าง mapping
print("\n📋 ตัวอย่าง ETF IDs:")
for ticker in ['AGG', 'SPY', 'QQQ', 'VTI', 'BND']:
    if ticker in etf_map:
        print(f"   {ticker} → etf_id = {etf_map[ticker]}")

print("\n🎉 พร้อม Import Price History!")

---

## Cell 3: Import Price History

**⚠️ รันครั้งเดียวเท่านั้น!** ถ้ารันซ้ำจะเกิด duplicate error

ใช้เวลา **1-2 นาที**

In [ ]:
start_time = datetime.now()

print("=" * 70)
print("📥 Import Price History (208,700 rows)".center(70))
print("=" * 70)

try:
    # อ่านไฟล์ CSV
    print("\n📁 อ่านไฟล์ CSV...")
    df_price = pd.read_csv('data/etf_price_history.csv')
    print(f"   ✅ อ่านได้ {len(df_price):,} rows")
    
    # แปลง ticker → etf_id
    print("\n🔄 แปลง ticker → etf_id...")
    df_price['etf_id'] = df_price['ticker'].map(etf_map)
    df_price = df_price.dropna(subset=['etf_id'])
    print(f"   ✅ แปลงแล้ว {len(df_price):,} rows")
    
    # Import เป็น batch
    batch_size = 5000
    total_batches = (len(df_price) + batch_size - 1) // batch_size
    
    print(f"\n📥 กำลัง Import ({total_batches} batches)...\n")
    
    imported = 0
    for i in range(0, len(df_price), batch_size):
        batch = df_price.iloc[i:i+batch_size]
        
        # Prepare data
        data = [
            (
                int(row['etf_id']),
                row['date'],
                float(row['open']),
                float(row['high']),
                float(row['low']),
                float(row['close']),
                int(row['volume'])
            )
            for _, row in batch.iterrows()
        ]
        
        # Batch insert
        cursor.executemany("""
            INSERT INTO price_history
            (etf_id, price_date, open_price, high_price, low_price, close_price, volume)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        """, data)
        
        conn.commit()
        
        imported += len(batch)
        current_batch = i // batch_size + 1
        percent = (imported / len(df_price)) * 100
        
        # Progress bar
        bar_length = 40
        filled = int(bar_length * imported / len(df_price))
        bar = '█' * filled + '░' * (bar_length - filled)
        
        print(f"   [{bar}] {percent:5.1f}% | {imported:,}/{len(df_price):,} rows | Batch {current_batch}/{total_batches}", end='\r')
    
    print()  # New line
    
    # ========================================
    # สรุปผลลัพธ์
    # ========================================
    elapsed = (datetime.now() - start_time).total_seconds()
    
    print("\n" + "=" * 70)
    print("🎉 Import เสร็จสมบูรณ์!".center(70))
    print("=" * 70)
    print(f"\n⏱️  ใช้เวลา: {elapsed:.1f} วินาที ({elapsed/60:.1f} นาที)")
    print(f"📦 Import: {imported:,} rows")
    print("\n" + "=" * 70)
    
except mysql.connector.IntegrityError as e:
    print(f"\n\n⚠️  IntegrityError: {e}\n")
    print("💡 มีข้อมูลอยู่แล้ว! ถ้าต้องการ import ใหม่ รัน SQL นี้ก่อน:")
    print("""
    TRUNCATE TABLE price_history;
    """)
    print("จากนั้นรัน Cell 3 ใหม่")

except Exception as e:
    print(f"\n\n❌ Error: {e}\n")
    import traceback
    traceback.print_exc()

---

## Cell 4: ตรวจสอบผลลัพธ์

In [ ]:
print("=" * 70)
print("📊 ตรวจสอบข้อมูลทั้งหมด")
print("=" * 70)

# เช็คจำนวนทุกตาราง
tables = {
    'etf_master': 50,
    'benchmark_portfolios': 35,
    'benchmark_holdings': 114,
    'price_history': 208700
}

all_ok = True
for table, expected in tables.items():
    cursor.execute(f"SELECT COUNT(*) as cnt FROM {table}")
    count = cursor.fetchone()['cnt']
    
    threshold = expected * 0.95
    status = "✅" if count >= threshold else "⚠️"
    if count < threshold:
        all_ok = False
    
    print(f"{status} {table:25s}: {count:>10,} rows (expected ~{expected:,})")

print("=" * 70)

if all_ok:
    print("\n🎉 ทุกอย่างถูกต้อง!")
else:
    print("\n⚠️  บางตารางมีข้อมูลน้อยกว่าที่คาดหวัง")

# ========================================
# แสดงตัวอย่างข้อมูล
# ========================================

print("\n\n📋 ตัวอย่างข้อมูล Price History (10 rows ล่าสุด):")
print("=" * 70)

cursor.execute("""
    SELECT 
        e.ticker_symbol,
        ph.price_date,
        ph.close_price,
        ph.volume
    FROM price_history ph
    JOIN etf_master e ON ph.etf_id = e.etf_id
    ORDER BY ph.price_date DESC
    LIMIT 10
""")

df_sample = pd.DataFrame(cursor.fetchall())
if not df_sample.empty:
    print(df_sample.to_string(index=False))

print("\n" + "=" * 70)
print("✅ ทุกอย่างเสร็จสมบูรณ์!".center(70))
print("=" * 70)
print("\n🎉 พร้อมใช้งาน! เปิด main.ipynb หรือ analytics.ipynb ได้เลย\n")

# ปิดการเชื่อมต่อ
cursor.close()
conn.close()
print("✅ ปิดการเชื่อมต่อ MySQL แล้ว")

---

## 💡 Tips

### ถ้าต้องการลบและ import ใหม่:

รัน SQL นี้ใน MySQL Workbench:

```sql
TRUNCATE TABLE price_history;
```

จากนั้นรัน Cell 3 ใหม่

### ถ้าไม่พบไฟล์ CSV:

- ตรวจสอบว่า Jupyter Notebook เปิดอยู่ที่โฟลเดอร์ `desktop-tutorial`
- ตรวจสอบว่ามีไฟล์ `data/etf_price_history.csv`

### ถ้า import ช้าเกินไป:

- ปกติใช้เวลา 1-2 นาที
- ถ้าช้ากว่านี้ ลองเพิ่ม `batch_size` จาก 5000 เป็น 10000 ใน Cell 3